# TinyCeNN-LM — Compact 8-Shard Top-2 Routed FFN

This notebook keeps the Transformer fully removed and replaces the previous **8 full FFN experts** with a **parameter-neutral sharded FFN**:

- original dense CeNN SwiGLU inner width: **768**
- split into **8 shards × 96 channels**
- total shard parameters = **one original dense FFN**
- Top-2 router learns per-token specialization
- all 8 shards reconstruct the complete FFN, while the Top-2 path adds a routed correction
- warm start is exact when `route_mix = 0`
- default training budget: **20M tokens** with a **50-minute training cap**

The notebook resumes `TinyCeNN-LM-Distilled-v2`, publishes the best result as `TinyCeNN-LM-Sharded-MoE-Top2`, downloads it again, and reproduces the rigorous held-out benchmark.


In [ ]:
import subprocess, sys, pathlib, importlib

subprocess.run(["nvidia-smi"], check=False)

REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()
import tinycenn_lm
print("TinyCeNN-LM:", tinycenn_lm.__file__)


## Hugging Face login

Add a Hugging Face **write token** to Colab Secrets as `HF_TOKEN`.


In [ ]:
from huggingface_hub import HfApi, login, snapshot_download
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    login()
api = HfApi()
hf_user = api.whoami()["name"]
print("Logged in as:", hf_user)


In [ ]:
SOURCE_HF_REPO = f"{hf_user}/TinyCeNN-LM-Distilled-v2"
plain_student = snapshot_download(repo_id=SOURCE_HF_REPO, repo_type="model")
print("Warm-start checkpoint:", plain_student)


In [ ]:
MAX_TOKENS = 20_000_000
MAX_RUNTIME_MINUTES = 50
CONTEXT_LENGTH = 256
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
NUM_SHARDS = 8
TOP_K = 2
EVAL_BATCHES = 64
EVAL_BATCH_SIZE = 4
EVAL_EVERY = 500
SHUFFLE_BUFFER = 4096
OUTPUT_DIR = str(REPO_DIR / "checkpoints/cenn-sharded-moe-top2")
print("Dense FFN inner width:", 192 * 4)
print("Shard width:", (192 * 4) // NUM_SHARDS)
print("Held-out tokens:", EVAL_BATCHES * EVAL_BATCH_SIZE * CONTEXT_LENGTH)


In [ ]:
import shutil
for p in (pathlib.Path(OUTPUT_DIR), pathlib.Path(OUTPUT_DIR + "-best")):
    if p.exists():
        shutil.rmtree(p)
cmd = [
    sys.executable, str(REPO_DIR / "scripts/train_sharded_moe_distill.py"),
    "--warmstart-plain-dir", plain_student,
    "--max-tokens", str(MAX_TOKENS),
    "--max-runtime-minutes", str(MAX_RUNTIME_MINUTES),
    "--context-length", str(CONTEXT_LENGTH),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--learning-rate", str(LEARNING_RATE),
    "--num-shards", str(NUM_SHARDS),
    "--top-k", str(TOP_K),
    "--eval-batches", str(EVAL_BATCHES),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--eval-every", str(EVAL_EVERY),
    "--shuffle-buffer", str(SHUFFLE_BUFFER),
    "--output-dir", OUTPUT_DIR,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
import json
report_path = pathlib.Path(OUTPUT_DIR) / "sharded_moe_distillation_report.json"
report = json.loads(report_path.read_text())
summary = {
    "status": report["status"],
    "stop_reason": report["stop_reason"],
    "seen_tokens_this_run": report["seen_tokens_this_run"],
    "cumulative_training_tokens": report["cumulative_training_tokens"],
    "trainable_parameters": report["parameters"]["trainable"],
    "plain_cenn_trainable": report["parameters"]["plain_cenn_trainable"],
    "parameter_overhead_percent": report["parameters"]["extra_over_plain_percent"],
    "warmstart_ce_delta": report["warmstart_ce_delta"],
    "run_start_ce": report["run_start"]["student_ce"],
    "best_student_ce": report["best"]["student_ce"],
    "teacher_ce": report["best"]["teacher_ce"],
    "best_student_ppl": report["best"]["student_ppl"],
    "teacher_ppl": report["best"]["teacher_ppl"],
    "teacher_gap_recovery_percent": 100 * report["teacher_gap_recovery_fraction"],
    "route_mix": report["best"]["route_mix"],
    "shard_fraction": report["best"]["shard_fraction"],
    "elapsed_training_minutes": report["elapsed_training_seconds"] / 60,
}
print(json.dumps(summary, indent=2))


In [ ]:
best_dir = pathlib.Path(OUTPUT_DIR + "-best")
publish_dir = best_dir if best_dir.exists() else pathlib.Path(OUTPUT_DIR)
HF_MODEL_NAME = "TinyCeNN-LM-Sharded-MoE-Top2"
HF_REPO_ID = f"{hf_user}/{HF_MODEL_NAME}"
card = f'''---
base_model: arnir0/Tiny-LLM
library_name: transformers
pipeline_tag: text-generation
tags:
- cenn
- moe
- top-2-routing
- transformer-free
- knowledge-distillation
- parameter-efficient
---

# TinyCeNN-LM Sharded MoE Top-2

Transformer-free CeNN student using a parameter-neutral routed FFN.

- 8 shards × 96 inner channels
- Top-2 routing
- trainable parameters: {report["parameters"]["trainable"]:,}
- overhead vs plain CeNN: {report["parameters"]["extra_over_plain_percent"]:.3f}%
- held-out tokens: {report["evaluation"]["tokens"]:,}
- best CE: {report["best"]["student_ce"]:.6f}
- teacher CE: {report["best"]["teacher_ce"]:.6f}
- teacher-gap recovery: {100*report["teacher_gap_recovery_fraction"]:.2f}%
'''
(publish_dir / "README.md").write_text(card, encoding="utf-8")
(publish_dir / "sharded_moe_distillation_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
api.create_repo(HF_REPO_ID, repo_type="model", private=False, exist_ok=True)
api.upload_folder(repo_id=HF_REPO_ID, repo_type="model", folder_path=str(publish_dir), commit_message=f"Compact sharded Top-2 CeNN: CE {report['best']['student_ce']:.4f}")
print(f"https://huggingface.co/{HF_REPO_ID}")


In [ ]:
parity_cmd = [sys.executable, str(REPO_DIR / "scripts/eval_sharded_moe.py"), "--hf-repo", HF_REPO_ID, "--ce-tolerance", "0.02"]
subprocess.run(parity_cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
import torch
from transformers import AutoTokenizer
from tinycenn_lm import build_sharded_moe_student, ShardedMoECeNNReplacementLayer
downloaded = snapshot_download(repo_id=HF_REPO_ID, repo_type="model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else (torch.float16 if device.type == "cuda" else torch.float32)
tokenizer = AutoTokenizer.from_pretrained(downloaded)
student = build_sharded_moe_student(downloaded, device=device, dtype=dtype).eval()
layers = [m for m in student.modules() if isinstance(m, ShardedMoECeNNReplacementLayer)]
assert len(layers) == 1
assert not any("self_attn" in name or ".mlp" in name for name, _ in student.named_modules())
layer = layers[0]
assert layer.config.num_shards == 8 and layer.config.top_k == 2 and layer.config.shard_inner == 96
print("Transformer-free structure: PASS")
print("8 × 96 sharded FFN: PASS")
print("Top-2 router: PASS")
for prompt in ["The capital of Austria is", "Artificial intelligence can help", "A small language model", "In the future, efficient AI"]:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        output = student.generate(**inputs, max_new_tokens=40, do_sample=False, use_cache=False, pad_token_id=tokenizer.eos_token_id)
    print("\nPROMPT:", prompt)
    print(tokenizer.decode(output[0], skip_special_tokens=True))
